# PS-Q1 Biomedical Signal Classifier — Phase 2

QFF 2026 QML Track. This notebook reproduces the key results.

In [ ]:
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, "src")
sys.path.insert(0, "benchmark")
from data_utils import load_data, fit_preprocess, transform_preprocess
from classical_baseline import train_classical
from quantum_vqc import train_vqc, evaluate_vqc, N_QUBITS
from qff_qml_benchmark import task_a_bell
print("Imports OK")


## Task A — Bell Fidelity

In [ ]:
res_a = task_a_bell()
print("Ideal F:", res_a["ideal"]["fidelity"])
print("Noisy F:", res_a["noisy"]["fidelity"], "±", res_a["noisy"]["fidelity_se"])


## Classical Baseline

In [ ]:
X_train, y_train, X_val, y_val, X_test, sample_ids, _ = load_data()
best_model, best_name, classical_res = train_classical(X_train, y_train, X_val, y_val)
print("Best:", best_name, "macro-F1:", classical_res[best_name]["macro_f1"])


## Quantum VQC (subset for speed)

In [ ]:
X_tr_p, transformers = fit_preprocess(X_train, n_components=N_QUBITS)
X_va_p = transform_preprocess(X_val, transformers)
theta, hist, ttime, npar = train_vqc(X_tr_p, y_train, maxiter=10, n_sub=80, batch_size=8)
print("Train time:", round(ttime,1), "params:", npar)
idx = np.random.RandomState(42).choice(len(X_va_p), 60, replace=False)
ideal = evaluate_vqc(theta, X_va_p[idx], y_val[idx], noisy=False, shots=256)
print("Ideal subset macro-F1:", round(ideal["macro_f1"], 4))


## Predictions & Validation

In [ ]:
preds = best_model.predict(X_test)
df = pd.DataFrame({"sample_id": sample_ids, "label": preds})
df.to_csv("predictions_q1.csv", index=False)
print("Wrote predictions_q1.csv", df.shape)
